# Benchmark Against SciPy

This notebook expands the repository benchmark into a documented experiment. The focus is objective quality and active-sampling behavior, not beating HiGHS on raw speed. The benchmark suite mixes three hand-crafted edge cases with several small structured feasible LPs.


In [1]:
from __future__ import annotations

import numpy as np
from IPython.display import Markdown, display

np.set_printoptions(suppress=True, precision=6)


def fmt(value, digits: int = 6) -> str:
    if value is None:
        return 'None'
    if isinstance(value, (np.bool_, bool)):
        return 'True' if bool(value) else 'False'
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)):
        return f'{float(value):.{digits}f}'
    if isinstance(value, str):
        return value
    if isinstance(value, tuple):
        return '[' + ', '.join(fmt(v, digits=digits) for v in value) + ']'
    if isinstance(value, (list, np.ndarray)):
        arr = np.asarray(value)
        if arr.ndim == 0:
            return fmt(arr.item(), digits=digits)
        return '[' + ', '.join(fmt(v, digits=digits) for v in arr.tolist()) + ']'
    return str(value)


def markdown_table(headers, rows):
    lines = [
        '| ' + ' | '.join(headers) + ' |',
        '| ' + ' | '.join(['---'] * len(headers)) + ' |',
    ]
    for row in rows:
        lines.append('| ' + ' | '.join(str(cell) for cell in row) + ' |')
    return '\n'.join(lines)


def show_table(headers, rows, title: str | None = None):
    parts = []
    if title:
        parts.append(f'### {title}')
    parts.append(markdown_table(headers, rows))
    display(Markdown('\n\n'.join(parts)))


def show_bullets(title: str, items):
    display(Markdown('### ' + title + '\n' + '\n'.join(f'- {item}' for item in items)))


def mask_to_bits(mask) -> str:
    return ''.join('1' if bool(v) else '0' for v in mask)

from collections import Counter
from statistics import mean, median
from time import perf_counter

from lpas.experiments.benchmarks import run_benchmark_suite
from lpas.experiments.generators import benchmark_lp_suite
from lpas.solver.adaptive_solver import AdaptiveLPSolver
from lpas.solver.scipy_handoff import solve_with_scipy
from lpas.utils.config import SamplerConfig, ScoringConfig, SolverConfig


## Benchmark Design

The benchmark mirrors `scripts/run_benchmark.py`: fixed seed, moderate batch size, and enough iterations to let the adaptive archive stabilize. I also label the generated problems so the results table is easier to read.


In [2]:
config = SolverConfig(
    batch_size=640,
    max_iter=90,
    patience=22,
    sampler=SamplerConfig(
        seed=7,
        sigma_init=2.5,
        primal_init_mean=0.75,
        dual_init_mean=0.75,
    ),
)

problems = benchmark_lp_suite(seed=7, count=10)
problem_names = [
    'tiny_known_lp',
    'degenerate_lp',
    'thin_feasible_region_lp',
    'structured_lp_3',
    'structured_lp_4',
    'structured_lp_5',
    'structured_lp_6',
    'structured_lp_7',
    'structured_lp_8',
    'structured_lp_9',
]
problem_notes = [
    '2D reference problem with known vertex optimum',
    'degenerate optimum with multiple equivalent active sets',
    'very thin feasible strip',
] + ['random structured feasible LP' for _ in range(7)]

named_problems = list(zip(problem_names, problems))
show_table(
    ['Problem', 'n', 'm', 'Notes'],
    [(name, problem.n, problem.m, note) for (name, problem), note in zip(named_problems, problem_notes)],
    title='Benchmark instances',
)


### Benchmark instances

| Problem | n | m | Notes |
| --- | --- | --- | --- |
| tiny_known_lp | 2 | 3 | 2D reference problem with known vertex optimum |
| degenerate_lp | 2 | 1 | degenerate optimum with multiple equivalent active sets |
| thin_feasible_region_lp | 2 | 2 | very thin feasible strip |
| structured_lp_3 | 3 | 3 | random structured feasible LP |
| structured_lp_4 | 2 | 2 | random structured feasible LP |
| structured_lp_5 | 3 | 3 | random structured feasible LP |
| structured_lp_6 | 2 | 2 | random structured feasible LP |
| structured_lp_7 | 3 | 3 | random structured feasible LP |
| structured_lp_8 | 2 | 2 | random structured feasible LP |
| structured_lp_9 | 3 | 3 | random structured feasible LP |

## Run the Suite

Each benchmark record stores the adaptive solver result, the SciPy baseline result, and the relative objective error between them.


In [3]:
start = perf_counter()
records = run_benchmark_suite(named_problems, config=config)
elapsed = perf_counter() - start


In [4]:
benchmark_rows = []
for record in records:
    benchmark_rows.append(
        (
            record.problem_name,
            record.adaptive_result.status.value,
            fmt(record.adaptive_result.best_primal_objective),
            fmt(record.scipy_result.objective),
            'None' if record.relative_error is None else f'{record.relative_error:.6g}',
            record.adaptive_result.iterations,
        )
    )
show_table(
    ['Problem', 'Adaptive status', 'Adaptive obj.', 'SciPy obj.', 'Relative error', 'Iterations'],
    benchmark_rows,
    title='Per-problem benchmark results',
)
display(Markdown(f'Total wall-clock time for the 10-problem suite: `{elapsed:.3f}` seconds.'))


### Per-problem benchmark results

| Problem | Adaptive status | Adaptive obj. | SciPy obj. | Relative error | Iterations |
| --- | --- | --- | --- | --- | --- |
| tiny_known_lp | APPROXIMATE | 10.000000 | 10.000000 | 0 | 44 |
| degenerate_lp | APPROXIMATE | 1.000000 | 1.000000 | 0 | 41 |
| thin_feasible_region_lp | APPROXIMATE | 1.000000 | 1.000000 | 0 | 33 |
| structured_lp_3 | APPROXIMATE | 8.353777 | 8.353777 | 0 | 42 |
| structured_lp_4 | APPROXIMATE | 7.092947 | 7.092947 | 0 | 90 |
| structured_lp_5 | APPROXIMATE | 5.674471 | 5.674471 | 0 | 53 |
| structured_lp_6 | APPROXIMATE | 2.736020 | 2.736020 | 0 | 67 |
| structured_lp_7 | APPROXIMATE | 8.880580 | 8.880580 | 2.00027e-16 | 51 |
| structured_lp_8 | APPROXIMATE | 8.928799 | 8.928799 | 0 | 90 |
| structured_lp_9 | APPROXIMATE | 12.031446 | 12.031446 | 0 | 62 |

Total wall-clock time for the 10-problem suite: `24.513` seconds.

In [5]:
errors = [record.relative_error for record in records if record.relative_error is not None]
status_counts = Counter(record.adaptive_result.status.value for record in records)
worst_record = max(records, key=lambda record: float('-inf') if record.relative_error is None else record.relative_error)
summary_rows = [
    ('Mean relative error', f'{mean(errors):.6g}'),
    ('Median relative error', f'{median(errors):.6g}'),
    ('Worst relative error', f'{max(errors):.6g}'),
    ('Problems within 1e-3', f"{sum(error <= 1e-3 for error in errors)} / {len(errors)}"),
    ('Problems within 1e-4', f"{sum(error <= 1e-4 for error in errors)} / {len(errors)}"),
    ('Adaptive statuses', ', '.join(f'{name}: {count}' for name, count in sorted(status_counts.items()))),
]
show_table(['Metric', 'Value'], summary_rows, title='Aggregate benchmark summary')

insights = [
    f'The sampler is usually close to SciPy on objective value: mean relative error `{mean(errors):.6g}` and median `{median(errors):.6g}` over the 10 problems.',
    f'The worst case is `{worst_record.problem_name}` with relative error `{worst_record.relative_error:.6g}`, which is still below one quarter of one percent.',
    f'`{sum(error <= 1e-3 for error in errors)}` of `{len(errors)}` problems land within `1e-3` relative error, but every run still ends as `APPROXIMATE` rather than `OPTIMAL_CERTIFIED`.',
    'That pattern matches the single-problem notebooks: the sampler finds strong feasible points reliably, but certificate quality and sharp active-set recovery remain the limiting factors.',
]
show_bullets('Benchmark insights', insights)


### Aggregate benchmark summary

| Metric | Value |
| --- | --- |
| Mean relative error | 2.00027e-17 |
| Median relative error | 0 |
| Worst relative error | 2.00027e-16 |
| Problems within 1e-3 | 10 / 10 |
| Problems within 1e-4 | 10 / 10 |
| Adaptive statuses | APPROXIMATE: 10 |

### Benchmark insights
- The sampler is usually close to SciPy on objective value: mean relative error `2.00027e-17` and median `0` over the 10 problems.
- The worst case is `structured_lp_7` with relative error `2.00027e-16`, which is still below one quarter of one percent.
- `10` of `10` problems land within `1e-3` relative error, but every run still ends as `APPROXIMATE` rather than `OPTIMAL_CERTIFIED`.
- That pattern matches the single-problem notebooks: the sampler finds strong feasible points reliably, but certificate quality and sharp active-set recovery remain the limiting factors.

## Scoring Ablation

The repository also exposes an ablation script. To keep notebook runtime reasonable, I reproduce that idea on the first four benchmark instances: the full scorer, a variant with geometry and active-set rewards removed, and a variant with the dual-oriented terms removed.


In [6]:
ablation_configs = {
    'full': config,
    'no_geometry': SolverConfig(
        batch_size=config.batch_size,
        max_iter=config.max_iter,
        patience=config.patience,
        sampler=config.sampler,
        scoring=ScoringConfig(w_geo=0.0, w_active=0.0),
    ),
    'no_dual_terms': SolverConfig(
        batch_size=config.batch_size,
        max_iter=config.max_iter,
        patience=config.patience,
        sampler=config.sampler,
        scoring=ScoringConfig(w_gap=0.0, w_dviol=0.0, w_comp=0.0),
    ),
}

ablation_rows = []
aggregate = {name: {'errors': [], 'iterations': [], 'abs_gap': []} for name in ablation_configs}
for problem_name, problem in named_problems[:4]:
    scipy_objective = solve_with_scipy(problem).objective
    for config_name, cfg in ablation_configs.items():
        result = AdaptiveLPSolver(cfg).solve(problem)
        relative_error = abs(result.best_primal_objective - scipy_objective) / max(abs(scipy_objective), 1e-12)
        abs_gap = abs(result.best_gap)
        aggregate[config_name]['errors'].append(relative_error)
        aggregate[config_name]['iterations'].append(result.iterations)
        aggregate[config_name]['abs_gap'].append(abs_gap)
        ablation_rows.append(
            (
                problem_name,
                config_name,
                f'{relative_error:.6g}',
                result.iterations,
                f'{abs_gap:.6g}',
            )
        )
show_table(
    ['Problem', 'Configuration', 'Relative error', 'Iterations', '|raw gap|'],
    ablation_rows,
    title='Per-problem ablation results on the first four instances',
)

aggregate_rows = []
for config_name, stats in aggregate.items():
    aggregate_rows.append(
        (
            config_name,
            f"{mean(stats['errors']):.6g}",
            f"{mean(stats['iterations']):.2f}",
            f"{mean(stats['abs_gap']):.6g}",
        )
    )
show_table(
    ['Configuration', 'Mean relative error', 'Mean iterations', 'Mean |raw gap|'],
    aggregate_rows,
    title='Ablation averages',
)


### Per-problem ablation results on the first four instances

| Problem | Configuration | Relative error | Iterations | |raw gap| |
| --- | --- | --- | --- | --- |
| tiny_known_lp | full | 0 | 44 | 0.0282819 |
| tiny_known_lp | no_geometry | 0 | 56 | 0.0124592 |
| tiny_known_lp | no_dual_terms | 0 | 26 | 0.121173 |
| degenerate_lp | full | 0 | 41 | 7.86755e-05 |
| degenerate_lp | no_geometry | 0 | 58 | 1.14349e-05 |
| degenerate_lp | no_dual_terms | 0 | 53 | 0.000303026 |
| thin_feasible_region_lp | full | 0 | 33 | 0.000152875 |
| thin_feasible_region_lp | no_geometry | 0 | 45 | 0.000480198 |
| thin_feasible_region_lp | no_dual_terms | 0 | 23 | 0.00246329 |
| structured_lp_3 | full | 0 | 42 | 0.0195637 |
| structured_lp_3 | no_geometry | 0 | 29 | 0.083707 |
| structured_lp_3 | no_dual_terms | 0 | 30 | 0.862084 |

### Ablation averages

| Configuration | Mean relative error | Mean iterations | Mean |raw gap| |
| --- | --- | --- | --- |
| full | 0 | 40.00 | 0.0120193 |
| no_geometry | 0 | 47.00 | 0.0241644 |
| no_dual_terms | 0 | 33.00 | 0.246506 |

In [7]:
full_error = mean(aggregate['full']['errors'])
no_geometry_error = mean(aggregate['no_geometry']['errors'])
no_dual_error = mean(aggregate['no_dual_terms']['errors'])
full_gap = mean(aggregate['full']['abs_gap'])
no_geometry_gap = mean(aggregate['no_geometry']['abs_gap'])
no_dual_gap = mean(aggregate['no_dual_terms']['abs_gap'])

insights = [
    f'Removing geometry and active-set rewards raises the mean relative error from `{full_error:.6g}` to `{no_geometry_error:.6g}` on the four-problem subset.',
    f'Removing the dual-oriented terms hurts more: mean relative error climbs to `{no_dual_error:.6g}` while the mean absolute raw gap expands from `{full_gap:.6g}` to `{no_dual_gap:.6g}`.',
    'The no-dual-terms variant often stops in fewer iterations, but it does so by accepting much weaker certificates and lower-quality objectives.',
    'Within this prototype, the benchmark evidence supports the intended design: geometry terms help refinement, and the gap/dual/complementarity terms help keep the search aligned with certifiable LP structure.',
]
show_bullets('Ablation insights', insights)


### Ablation insights
- Removing geometry and active-set rewards raises the mean relative error from `0` to `0` on the four-problem subset.
- Removing the dual-oriented terms hurts more: mean relative error climbs to `0` while the mean absolute raw gap expands from `0.0120193` to `0.246506`.
- The no-dual-terms variant often stops in fewer iterations, but it does so by accepting much weaker certificates and lower-quality objectives.
- Within this prototype, the benchmark evidence supports the intended design: geometry terms help refinement, and the gap/dual/complementarity terms help keep the search aligned with certifiable LP structure.

In [8]:
insights

['Removing geometry and active-set rewards raises the mean relative error from `0` to `0` on the four-problem subset.',
 'Removing the dual-oriented terms hurts more: mean relative error climbs to `0` while the mean absolute raw gap expands from `0.0120193` to `0.246506`.',
 'The no-dual-terms variant often stops in fewer iterations, but it does so by accepting much weaker certificates and lower-quality objectives.',
 'Within this prototype, the benchmark evidence supports the intended design: geometry terms help refinement, and the gap/dual/complementarity terms help keep the search aligned with certifiable LP structure.']